# [Optional] Introduction to Strands Agents

This optional module introduces the Strands Agents SDK for building AI agents. Complete this if you're new to agentic AI development before proceeding to the main Developer Journey labs.

## Learning Objectives

By the end of this notebook, you will be able to:
- Create a basic agent using the Strands Agents SDK
- Add built-in and custom tools to extend agent capabilities
- Understand the agent execution loop and metrics
- Deploy an agent to Amazon Bedrock AgentCore Runtime

**Duration**: ~30 minutes

## What is Strands Agents?

[Strands Agents](https://strandsagents.com/) is an open-source SDK for building AI agents with a model-driven approach. Instead of manually orchestrating complex workflows, Strands lets the model decide when and how to use tools.

### Core Components

| Component | Description |
|-----------|-------------|
| **Model** | The LLM that powers reasoning (e.g., Claude on Bedrock) |
| **System Prompt** | Instructions defining agent behavior and personality |
| **Tools** | Functions the agent can call to take actions |

### Agent Loop Architecture

![Strands Agent Architecture](images/strands-agent-architecture.png)

The agent autonomously decides whether to respond directly or use tools based on the user's request.

---

## 1. Setup

Configure AWS credentials and model settings.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Configuration
REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
MODEL_ID = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"

print(f"Region: {REGION}")
print(f"Model: {MODEL_ID}")

Region: us-east-1
Model: global.anthropic.claude-sonnet-4-5-20250929-v1:0


---

## 2. Your First Agent

Create a simple agent with just a few lines of code. The agent uses Claude on Amazon Bedrock for reasoning.

In [2]:
from strands import Agent

# Create a basic agent
agent = Agent(
    model=MODEL_ID,
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

# Invoke the agent
agent("What is Amazon Bedrock in one sentence?")

Amazon Bedrock is a fully managed AWS service that provides access to foundation models from leading AI companies through a single API for building and scaling generative AI applications.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Amazon Bedrock is a fully managed AWS service that provides access to foundation models from leading AI companies through a single API for building and scaling generative AI applications.'}]}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[2.619097948074341], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='cc65b381-148d-476f-b3f6-e16323e9d5f0', usage={'inputTokens': 29, 'outputTokens': 36, 'totalTokens': 65})], usage={'inputTokens': 29, 'outputTokens': 36, 'totalTokens': 65})], traces=[<strands.telemetry.metrics.Trace object at 0x110e42010>], accumulated_usage={'inputTokens': 29, 'outputTokens': 36, 'totalTokens': 65}, accumulated_metrics={'latencyMs': 1824}), state={}, interrupts=None, structured_output=None)

<div class="alert alert-block alert-info">
<b>Note:</b> The agent automatically handles the conversation with the model. You simply call the agent like a function with your input.
</div>

---

## 3. Adding Tools

Tools extend agent capabilities beyond text generation. The agent autonomously decides when to use tools based on the user's request.

### Built-in Tools

Strands provides common tools out of the box via the `strands-agents-tools` package:

In [3]:
from strands import Agent
from strands_tools import calculator, current_time

# Create agent with built-in tools
agent_with_tools = Agent(
    model=MODEL_ID,
    tools=[calculator, current_time],
    system_prompt="You are a helpful assistant with access to a calculator and clock."
)

# The agent automatically uses the calculator tool
agent_with_tools("What is 15% of 250?")


Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 0.15 * 250          │                                                                            │
│  │ Result    │ 37.5                │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

15% of 250 is **37.5**.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '15% of 250 is **37.5**.'}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_1lNoXjuLizAAfM2CxqXIdd', 'name': 'calculator', 'input': {'expression': '0.15 * 250'}}, call_count=1, success_count=1, error_count=0, total_time=0.008791923522949219)}, cycle_durations=[1.7309069633483887], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='eb8003c5-c69a-4439-848b-ca429da028db', usage={'inputTokens': 2052, 'outputTokens': 57, 'totalTokens': 2109}), EventLoopCycleMetric(event_loop_cycle_id='aac7f135-5ec5-48c1-8daf-6101ba234ade', usage={'inputTokens': 2128, 'outputTokens': 15, 'totalTokens': 2143})], usage={'inputTokens': 4180, 'outputTokens': 72, 'totalTokens': 4252})], traces=[<strands.telemetry.metrics.Trace object at 0x1121834d0>, <strands.telemetry.metrics.Trace object at 0x112b20350>], accumulated_usage={'inputT

In [4]:
# The agent uses the current_time tool
agent_with_tools("What time is it now?")


Tool #2: current_time
The current time is **3:30:16 AM UTC** on March 15, 2026.

If you'd like to know the time in a different timezone, just let me know which one!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "The current time is **3:30:16 AM UTC** on March 15, 2026.\n\nIf you'd like to know the time in a different timezone, just let me know which one!"}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_1lNoXjuLizAAfM2CxqXIdd', 'name': 'calculator', 'input': {'expression': '0.15 * 250'}}, call_count=1, success_count=1, error_count=0, total_time=0.008791923522949219), 'current_time': ToolMetrics(tool={'toolUseId': 'tooluse_0duVutcf8NfJoVhXT4LtXx', 'name': 'current_time', 'input': {}}, call_count=1, success_count=1, error_count=0, total_time=0.0006330013275146484)}, cycle_durations=[1.7309069633483887, 2.154650926589966], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='eb8003c5-c69a-4439-848b-ca429da028db', usage={'inputTokens': 2052, 'outputTokens': 57, 'totalTokens': 2109}), EventLoopCycleMetric(event_loop_cy

### Custom Tools

Create domain-specific tools using the `@tool` decorator. This is essential for building agents that interact with your business systems.

In [5]:
from strands import Agent, tool

@tool
def lookup_order(order_id: str) -> str:
    """Look up order status by order ID.
    
    Args:
        order_id: The order ID to look up (e.g., ORD-12345)
    
    Returns:
        Order status information
    """
    # Simulated order database
    orders = {
        "ORD-12345": {"status": "Shipped", "eta": "2 days", "carrier": "FedEx"},
        "ORD-67890": {"status": "Processing", "eta": "5 days", "carrier": "Pending"},
    }
    
    if order_id in orders:
        order = orders[order_id]
        return f"Order {order_id}: {order['status']}, ETA: {order['eta']}, Carrier: {order['carrier']}"
    return f"Order {order_id} not found"


@tool
def check_inventory(product_sku: str) -> str:
    """Check inventory for a product SKU.
    
    Args:
        product_sku: The product SKU to check
    
    Returns:
        Inventory status
    """
    # Simulated inventory
    inventory = {
        "SKU-KB-001": {"name": "Wireless Keyboard", "qty": 150, "warehouse": "Seattle"},
        "SKU-MS-002": {"name": "Gaming Mouse", "qty": 0, "warehouse": "N/A"},
    }
    
    if product_sku in inventory:
        item = inventory[product_sku]
        status = "In Stock" if item['qty'] > 0 else "Out of Stock"
        return f"{item['name']}: {status} ({item['qty']} units in {item['warehouse']})"
    return f"Product {product_sku} not found"


print("Custom tools defined: lookup_order, check_inventory")

Custom tools defined: lookup_order, check_inventory


<div class="alert alert-block alert-warning">
<b>Key Insight:</b> The docstring in your tool function is critical - it tells the model what the tool does and when to use it. Write clear, descriptive docstrings for better tool selection.
</div>

---

## 4. Customer Support Agent Example

Let's build a customer support agent similar to what we'll use in the main Developer Journey labs.

In [6]:
support_agent = Agent(
    model=MODEL_ID,
    tools=[lookup_order, check_inventory],
    system_prompt="""You are a customer support agent for CloudCommerce.

Your responsibilities:
- Help customers check order status using the lookup_order tool
- Check product inventory using the check_inventory tool
- Be friendly, professional, and helpful

Always use the available tools to get accurate information.
If you cannot find information, apologize and offer alternatives."""
)

print("Customer support agent created with 2 tools")

Customer support agent created with 2 tools


In [7]:
# Test order lookup
support_agent("Where is my order ORD-12345?")


Tool #1: lookup_order
Great news! I found your order information.

**Order ORD-12345:**
- **Status:** Shipped ✓
- **Estimated Arrival:** 2 days
- **Carrier:** FedEx

Your order is on its way and should arrive within the next 2 days. You should receive a tracking number from FedEx that will allow you to track your package in real-time. Is there anything else I can help you with regarding this order?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Great news! I found your order information.\n\n**Order ORD-12345:**\n- **Status:** Shipped ✓\n- **Estimated Arrival:** 2 days\n- **Carrier:** FedEx\n\nYour order is on its way and should arrive within the next 2 days. You should receive a tracking number from FedEx that will allow you to track your package in real-time. Is there anything else I can help you with regarding this order?'}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'tooluse_XqexV7z4SPC8y4uaHZsd9f', 'name': 'lookup_order', 'input': {'order_id': 'ORD-12345'}}, call_count=1, success_count=1, error_count=0, total_time=0.0007009506225585938)}, cycle_durations=[2.9043009281158447], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='d92c83b8-198b-4480-8c9f-e06466441502', usage={'inputTokens': 766, 'outputTokens': 60, 'totalTokens': 826}), EventLoopCy

In [8]:
# Test inventory check
support_agent("Do you have the wireless keyboard SKU-KB-001 in stock?")


Tool #2: check_inventory
Yes, we do! The wireless keyboard (SKU-KB-001) is currently in stock.

**Product:** Wireless Keyboard
**SKU:** SKU-KB-001
**Status:** In Stock ✓
**Availability:** 150 units available in our Seattle warehouse

You're all set to place an order for this item whenever you're ready! Is there anything else I can help you with today?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Yes, we do! The wireless keyboard (SKU-KB-001) is currently in stock.\n\n**Product:** Wireless Keyboard\n**SKU:** SKU-KB-001\n**Status:** In Stock ✓\n**Availability:** 150 units available in our Seattle warehouse\n\nYou're all set to place an order for this item whenever you're ready! Is there anything else I can help you with today?"}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'tooluse_XqexV7z4SPC8y4uaHZsd9f', 'name': 'lookup_order', 'input': {'order_id': 'ORD-12345'}}, call_count=1, success_count=1, error_count=0, total_time=0.0007009506225585938), 'check_inventory': ToolMetrics(tool={'toolUseId': 'tooluse_Alv3axxx7MBFGDjgsKQwRN', 'name': 'check_inventory', 'input': {'product_sku': 'SKU-KB-001'}}, call_count=1, success_count=1, error_count=0, total_time=0.0002951622009277344)}, cycle_durations=[2.9043009281158447, 2.85827374458313], agen

In [9]:
# Test handling unknown order
support_agent("What's the status of order ORD-99999?")


Tool #3: lookup_order
I'm sorry, but I couldn't find any order with the ID **ORD-99999** in our system. 

This could mean:
- The order number might be incorrect or mistyped
- The order may not exist yet
- There could be a delay in the system updating

**Here's what you can do:**
- Double-check your order confirmation email for the correct order number
- If you just placed the order, please wait a few minutes and try again
- Contact our support team directly if you continue to have issues

Is there anything else I can help you with, or would you like me to look up a different order number?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I'm sorry, but I couldn't find any order with the ID **ORD-99999** in our system. \n\nThis could mean:\n- The order number might be incorrect or mistyped\n- The order may not exist yet\n- There could be a delay in the system updating\n\n**Here's what you can do:**\n- Double-check your order confirmation email for the correct order number\n- If you just placed the order, please wait a few minutes and try again\n- Contact our support team directly if you continue to have issues\n\nIs there anything else I can help you with, or would you like me to look up a different order number?"}]}, metrics=EventLoopMetrics(cycle_count=6, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'tooluse_jNHTnzoJcXyowDVlLY6Py5', 'name': 'lookup_order', 'input': {'order_id': 'ORD-99999'}}, call_count=2, success_count=2, error_count=0, total_time=0.001219034194946289), 'check_inventory': ToolMetrics(tool={'toolUse

---

## 5. Understanding Agent Execution

After running an agent, you can inspect execution metrics to understand what happened.

In [10]:
result = support_agent("Check if SKU-MS-002 is available")

# Access metrics from the result
print("\n" + "=" * 50)
print("EXECUTION METRICS")
print("=" * 50)

if hasattr(result, 'metrics'):
    metrics = result.metrics
    if hasattr(metrics, 'accumulated_usage'):
        usage = metrics.accumulated_usage
        print(f"Input tokens:  {usage.get('inputTokens', 'N/A')}")
        print(f"Output tokens: {usage.get('outputTokens', 'N/A')}")
        print(f"Total tokens:  {usage.get('totalTokens', 'N/A')}")
else:
    print("Metrics not available in this response format")


Tool #4: check_inventory
Unfortunately, the Gaming Mouse (SKU-MS-002) is currently out of stock.

**Product:** Gaming Mouse
**SKU:** SKU-MS-002
**Status:** Out of Stock ✗
**Availability:** 0 units available

I apologize for the inconvenience. Here are some options:
- Check back in a few days as we regularly restock popular items
- Contact our sales team to inquire about expected restock dates
- Ask if there's a similar product available that might meet your needs

Would you like me to check inventory for any other products, or is there anything else I can assist you with?
EXECUTION METRICS
Input tokens:  9096
Output tokens: 728
Total tokens:  9824


---

## 6. Deploy to Amazon Bedrock AgentCore Runtime

Amazon Bedrock AgentCore Runtime provides a managed, serverless environment for running agents in production.

![AgentCore Runtime Architecture](images/agentcore-runtime-architecture.png)

### What is AgentCore Runtime?

AgentCore Runtime is a fully managed service that hosts your AI agents:

1. **Packages your code** into a container
2. **Creates AWS resources** (IAM roles, ECR, etc.)
3. **Hosts your agent** in a serverless environment
4. **Provides an API endpoint** for invocation

### Key Benefits

| Feature | Description |
|---------|-------------|
| **Serverless** | No infrastructure to manage |
| **Auto-scaling** | Handles variable load automatically |
| **Session Isolation** | Each conversation runs in isolated context |
| **Framework Agnostic** | Works with Strands, LangGraph, CrewAI, and more |

### Step 1: Create Agent File

Create a file called `demo_agent.py` with your agent wrapped in `BedrockAgentCoreApp`:

In [11]:
%%writefile demo_agent.py
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool

app = BedrockAgentCoreApp()

@tool
def lookup_order(order_id: str) -> str:
    """Look up order status by order ID."""
    orders = {
        "ORD-12345": {"status": "Shipped", "eta": "2 days", "carrier": "FedEx"},
        "ORD-67890": {"status": "Processing", "eta": "5 days", "carrier": "Pending"},
    }
    if order_id in orders:
        order = orders[order_id]
        return f"Order {order_id}: {order['status']}, ETA: {order['eta']}, Carrier: {order['carrier']}"
    return f"Order {order_id} not found"

agent = Agent(
    model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    tools=[lookup_order],
    system_prompt="You are a helpful customer support agent."
)

@app.entrypoint
def invoke(payload):
    response = agent(payload.get("prompt", ""))
    return {"result": response.message}

if __name__ == "__main__":
    app.run()

Overwriting demo_agent.py


In [12]:
%%writefile requirements-demo.txt
bedrock-agentcore
strands-agents

Overwriting requirements-demo.txt


### Step 2: Configure and Deploy

Use the AgentCore Starter Toolkit to configure and deploy the agent:

In [13]:
from bedrock_agentcore_starter_toolkit import Runtime

# Initialize the Runtime
agentcore_runtime = Runtime()
AGENT_NAME = "demo_agent"

# Configure the agent
agentcore_runtime.configure(
    entrypoint="demo_agent.py",
    requirements_file="requirements-demo.txt",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=REGION,
    agent_name=AGENT_NAME
)

print("Agent configured")

/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
Entrypoint parsed: file=/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/demo_agent.py, bedrock_agentcore_name=demo_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: demo_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: 
/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/Dockerfile

Generated .dockerignore: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.dockerignore
Keeping 'demo_agent' as default agent
Bedrock AgentCore configured: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.bedrock_agentcore.yaml


Agent configured


### Step 3: Deploy and Test

Deploy the agent and test it:

In [14]:
# Deploy to AgentCore Runtime
# This builds the container, pushes to ECR, and creates the runtime
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print(f"Agent deployed: {launch_result.agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'demo_agent' to account 739907928487 (us-east-1)
Generated image tag: 20260315-033043-686
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: demo_agent
ECR repository available: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-demo_agent
Getting or creating execution role for agent: demo_agent
Using AWS region: us-east-1, account ID: 739907928487
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba


✅ Reusing existing ECR repository: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-demo_agent


✅ Reusing existing execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
Execution role available: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: demo_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-20b5f127ba
Reusing existing CodeBuild execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-20b5f127ba
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: demo_agent/source.zip
Updated CodeBuild project: bedrock-agentcore-demo_agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.3s
🔄 PROVISIONING started (total: 2s)
✅ PROVISIONING completed in 6.5s
🔄 DOWNLOAD_SOURCE started (total: 8s)
✅ DOWNLOAD_SOURCE complete

Agent deployed: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/demo_agent-AgI5IC4XgQ


In [15]:
import time

# Wait for agent to be ready
status = agentcore_runtime.status()
agent_status = status.endpoint["status"]
terminal_states = {"READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"}

while agent_status not in terminal_states:
    print(f"Agent status: {agent_status}")
    time.sleep(10)
    status = agentcore_runtime.status()
    agent_status = status.endpoint["status"]

print(f"Final status: {agent_status}")

Retrieved Bedrock AgentCore status for: demo_agent


Final status: READY


In [ ]:
import os
import json
import uuid
import boto3

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
data_client = boto3.client("bedrock-agentcore", region_name=region)

# Get agent ARN from deployment result or runtime status
try:
    agent_arn = launch_result.agent_arn
except NameError:
    status = agentcore_runtime.status()
    agent_arn = status.runtime["agentRuntimeArn"]


def invoke_agent(prompt):
    """Invoke the agent via AgentCore API."""
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))


# Test the deployed agent
print(f"Agent ARN: {agent_arn}")
print(invoke_agent("where is my order ORD-12345?"))

In [ ]:
# Clean up AgentCore resources using Python API
agentcore_runtime.destroy(delete_ecr_repo=True)

# Clean up local files
import os
for f in ['demo_agent.py', 'requirements-demo.txt', '.bedrock_agentcore.yaml', 'Dockerfile', '.dockerignore']:
    if os.path.exists(f):
        os.remove(f)

print("Cleanup complete!")

---

## Summary

| Concept | Description |
|---------|-------------|
| **Agent** | Model + System Prompt + Tools |
| **Tools** | Functions the agent can call |
| **@tool decorator** | Creates custom tools from Python functions |
| **Agent Loop** | Reason -> Select Tool -> Execute -> Repeat/Respond |
| **AgentCore Runtime** | Managed serverless environment for production |

### Key Takeaways

1. **Simple API**: Create agents with just 3 components (model, prompt, tools)
2. **Tool-driven**: The model decides when and how to use tools
3. **Custom tools**: Use `@tool` decorator for domain-specific functionality
4. **Production-ready**: AgentCore Runtime handles scaling and infrastructure

---

> **Note:** The tools used in this introduction (`lookup_order`, `check_inventory`) are different from the tools used in the main Developer Journey labs (`get_return_policy`, `get_product_info`, `get_technical_support`). The concepts you learned here — creating agents, defining custom tools, and deploying to AgentCore — apply the same way throughout the workshop.

**Next**: Continue to the main Developer Journey labs to build a complete customer support system with observability and evaluation.